# Pregunta 3 — Análisis exploratorio de Caltech Camera Traps

Cinco preguntas sobre los datos, cada una ligada a un entregable:

1. **¿Cómo están repartidas las particiones?** — define el experimento de
   generalización (entregable 3).
2. **¿Qué tan desbalanceadas están las especies?** — entregable 2.
3. **¿Qué tamaño tienen los animales en el cuadro?** — entregable 4.
4. **¿Las cámaras nuevas tienen otra mezcla de especies?** — es el
   confundidor que hay que controlar antes de atribuir nada a la ubicación.
5. **¿Cómo se ven las imágenes?** — de noche, de día, con el animal chico
   y lejos.

Todas las celdas **solo leen** disco. Ninguna entrena nada.

In [ ]:
import collections
import json
import pathlib
import sys

import numpy as np
import matplotlib.pyplot as plt
import yaml

# Raiz del proyecto en la maquina local. Orden: la variable REDES_RAIZ, la
# carpeta actual o alguna superior, y por ultimo la ruta fija de esta PC. Asi
# funciona en Jupyter/VS Code y en Colab conectado a un entorno de ejecucion
# local, aunque Jupyter se haya lanzado desde otra carpeta.
import os
RAIZ_LOCAL = pathlib.Path("D:/Backend/redesneuronales")
RAIZ = pathlib.Path(os.environ.get("REDES_RAIZ", pathlib.Path.cwd()))
while not (RAIZ / "pregunta1").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
if not (RAIZ / "pregunta1").is_dir():
    RAIZ = RAIZ_LOCAL
if not (RAIZ / "pregunta1").is_dir():
    raise FileNotFoundError(
        f"No encuentro el proyecto (busque desde {pathlib.Path.cwd()} y en "
        f"{RAIZ_LOCAL}). En Colab: Conectar -> 'Conectar a un entorno de "
        "ejecucion local', o define REDES_RAIZ con la ruta del proyecto.")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

P1 = RAIZ / "pregunta3/paso1_datos"
DS = RAIZ / "pregunta3/paso2_preprocesamiento/dataset"

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 9})

CLASES = yaml.safe_load((DS / "data_base.yaml").read_text(encoding="utf-8"))["names"]
if isinstance(CLASES, dict):
    CLASES = [CLASES[i] for i in sorted(CLASES)]
print(f"{len(CLASES)} clases: {CLASES}")


def lista(particion):
    """Las rutas de imagen de una particion."""
    p = DS / f"{particion}.txt"
    return p.read_text(encoding="utf-8").strip().splitlines() if p.is_file() else []


def cajas(ruta_imagen):
    """(clase, x, y, w, h) de la etiqueta YOLO que corresponde a una imagen."""
    p = pathlib.Path(ruta_imagen)
    lab = pathlib.Path(str(p).replace("/images/", "/labels/")
                       .replace("\\images\\", "\\labels\\")).with_suffix(".txt")
    if not lab.is_file():
        return []
    salida = []
    for linea in lab.read_text().strip().splitlines():
        partes = linea.split()
        if len(partes) >= 5:
            salida.append((int(partes[0]), *map(float, partes[1:5])))
    return salida

## 1. Las particiones ya son el experimento

`cis` y `trans` no son nombres administrativos: `cis_*` son **las mismas
cámaras** que vio el entrenamiento, `trans_*` son **cámaras nuevas**.

In [ ]:
resumen = {}
for p in ("train", "cis_val", "cis_test", "trans_val", "trans_test"):
    rutas = lista(p)
    n_cajas = 0
    clases = collections.Counter()
    for r in rutas:
        for c, *_ in cajas(r):
            n_cajas += 1
            clases[c] += 1
    resumen[p] = {"imagenes": len(rutas), "cajas": n_cajas, "clases": clases}
    print(f"{p:<12} {len(rutas):>7,} imagenes  {n_cajas:>7,} cajas  "
          f"{len(clases):>3} especies")

total = sum(v["imagenes"] for v in resumen.values())
print(f"\n  suma de las particiones: {total:,}")
etiquetas = len(list((P1/'labels').glob('*.txt'))) if (P1/'labels').is_dir() else 0
print(f"  archivos de etiqueta   : {etiquetas:,}")
print(f"  -> coinciden: las particiones son DISJUNTAS, "
      f"ninguna imagen aparece en dos.")

## 2. El desbalance de especies (entregable 2)

Este es el número que más condiciona la lectura del mAP.

In [ ]:
tr = resumen["train"]["clases"]
orden = [c for c, _ in tr.most_common()]
print(f"{'especie':<14} {'cajas train':>12} {'% del total':>12}")
print("-" * 42)
tot = sum(tr.values())
for c in orden:
    print(f"{CLASES[c]:<14} {tr[c]:>12,} {100*tr[c]/tot:>11.2f}%")
ausentes = [CLASES[i] for i in range(len(CLASES)) if i not in tr]
if ausentes:
    print(f"\n  SIN UNA SOLA CAJA en entrenamiento: {ausentes}")
print(f"\n  factor entre la mas comun y la mas rara: "
      f"{tr[orden[0]]/max(1,tr[orden[-1]]):.0f}x")

fig, ax = plt.subplots(figsize=(7, 3.6))
nom = [CLASES[c] for c in orden][::-1]
val = [tr[c] for c in orden][::-1]
ax.barh(nom, val, color=["#c0392b" if v < 100 else "#4c78a8" for v in val])
ax.set_xscale("log"); ax.set_xlabel("cajas en entrenamiento (escala log)")
ax.set_title("Desbalance de especies (rojo: menos de 100 cajas)")
plt.tight_layout(); plt.show()

### Por qué eso rompe el mAP macro

El mAP macro promedia las especies **por igual**. Una especie con un
puñado de cajas aporta un AP estimado con muchísimo ruido, y pesa lo mismo
que una con miles. Por eso el paso 4 reporta también el **mAP ponderado
por soporte**: las dos cifras juntas dicen algo que ninguna dice sola.

## 3. ¿Qué tamaño tienen los animales? (entregable 4)

In [ ]:
areas, areas_por_clase = [], collections.defaultdict(list)
for r in lista("cis_test"):
    for c, x, y, w, h in cajas(r):
        a = w * h            # fraccion del cuadro
        areas.append(a); areas_por_clase[c].append(a)
areas = np.array(areas)
print(f"{len(areas):,} cajas en cis_test")
print(f"  mediana del area: {100*np.median(areas):.2f}% del cuadro")
for lim, nombre in ((0.002, "diminuto  (<0.2%)"), (0.01, "chico  (<1%)"),
                    (0.05, "mediano (<5%)")):
    print(f"  {nombre:<20} {100*(areas < lim).mean():>5.1f}% de las cajas")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(100*areas[areas < 0.2], bins=80, color="#4c78a8")
axes[0].set_xlabel("% del cuadro que ocupa el animal")
axes[0].set_ylabel("cajas"); axes[0].set_title("Tamano del objeto")
axes[0].axvline(0.2, ls="--", c="crimson", lw=1)
axes[0].axvline(1.0, ls="--", c="orange", lw=1)

medianas = sorted(((np.median(v), CLASES[c]) for c, v in areas_por_clase.items()
                   if len(v) >= 20))
axes[1].barh([n for _, n in medianas], [100*m for m, _ in medianas],
             color="#4c78a8")
axes[1].set_xlabel("% del cuadro (mediana)")
axes[1].set_title("Tamano tipico por especie")
plt.tight_layout(); plt.show()

print("\n  La mayoria de los animales ocupa menos del 5% del cuadro. Es")
print("  la razon por la que la resolucion de entrada es una decision")
print("  real y no un default: a 640 px, un animal del 0.2% del cuadro")
print("  mide unos 9 pixeles de lado.")

## 4. El confundidor: ¿las cámaras nuevas tienen otra mezcla?

Si `trans_test` tuviera más animales difíciles, una caída del mAP no
probaría nada sobre la ubicación. Hay que mirarlo **antes** de concluir.

In [ ]:
cis, trans = resumen["cis_test"]["clases"], resumen["trans_test"]["clases"]
tc, tt = sum(cis.values()), sum(trans.values())
print(f"{'especie':<14} {'% cis_test':>11} {'% trans_test':>13} {'cambio':>9}")
print("-" * 52)
for c in sorted(set(cis) | set(trans),
                key=lambda k: -(trans.get(k, 0)/tt)):
    a, b = 100*cis.get(c, 0)/tc, 100*trans.get(c, 0)/tt
    print(f"{CLASES[c]:<14} {a:>10.2f}% {b:>12.2f}% {b-a:>+8.2f}")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
cl = sorted(set(cis) | set(trans), key=lambda k: -(trans.get(k, 0)/tt))
idx = np.arange(len(cl))
ax.bar(idx - 0.2, [100*cis.get(c, 0)/tc for c in cl], 0.4,
       label="cis_test (camaras vistas)")
ax.bar(idx + 0.2, [100*trans.get(c, 0)/tt for c in cl], 0.4,
       label="trans_test (camaras nuevas)")
ax.set_xticks(idx); ax.set_xticklabels([CLASES[c] for c in cl],
                                       rotation=65, fontsize=7)
ax.set_ylabel("% de las cajas"); ax.legend(fontsize=8)
ax.set_title("La mezcla de especies CAMBIA entre particiones")
plt.tight_layout(); plt.show()

print("\n  Las mezclas son distintas. Por eso el paso 5 remuestrea")
print("  trans_test para igualar la mezcla antes de medir la brecha:")
print("  sin ese control, la caida podria ser solo 'hay mas de las")
print("  dificiles' en vez de 'la camara es nueva'.")

## 5. Cómo se ven las imágenes

In [ ]:
rng = np.random.default_rng(0)
# Se eligen casos con animal chico, que son los dificiles.
candidatos = []
for r in rng.choice(lista("cis_test"), 400, replace=False):
    cj = cajas(r)
    if cj:
        candidatos.append((min(w*h for _, _, _, w, h in cj), r, cj))
candidatos.sort()
seleccion = [candidatos[0], candidatos[len(candidatos)//2], candidatos[-1]]

from PIL import Image
import matplotlib.patches as mp

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (area, ruta, cj) in zip(axes, seleccion):
    try:
        im = Image.open(ruta)
    except (FileNotFoundError, OSError):
        ax.text(.5, .5, "no se pudo abrir", ha="center"); ax.axis("off")
        continue
    W, H = im.size
    ax.imshow(im)
    for c, x, y, w, h in cj:
        ax.add_patch(mp.Rectangle(((x-w/2)*W, (y-h/2)*H), w*W, h*H,
                                  fill=False, ec="lime", lw=1.6))
        ax.text((x-w/2)*W, (y-h/2)*H-6, CLASES[c], color="lime", fontsize=7)
    ax.set_title(f"animal mas chico: {100*area:.2f}% del cuadro", fontsize=8)
    ax.axis("off"); ax.grid(False)
plt.tight_layout(); plt.show()
print("De izquierda a derecha: el caso dificil, el tipico y el facil.")

In [ ]:
# Dia contra noche: las camaras trampa disparan infrarrojo de noche, y la
# imagen pasa a ser monocroma. Es un cambio de dominio DENTRO del dataset.
nocturnas = diurnas = 0
for r in rng.choice(lista("cis_test"), 160, replace=False):
    try:
        a = np.asarray(Image.open(r).convert("RGB").resize((64, 48)),
                       dtype=np.float32)
    except (FileNotFoundError, OSError):
        continue
    # Una imagen infrarroja tiene los tres canales casi iguales.
    saturacion = (a.max(axis=2) - a.min(axis=2)).mean()
    if saturacion < 12:
        nocturnas += 1
    else:
        diurnas += 1
tot = nocturnas + diurnas
if tot:
    print(f"de {tot} imagenes muestreadas:")
    print(f"  monocromas (infrarrojo, nocturnas): {nocturnas} "
          f"({100*nocturnas/tot:.0f}%)")
    print(f"  en color (diurnas)                : {diurnas} "
          f"({100*diurnas/tot:.0f}%)")
    print("\n  Casi la mitad del dataset es infrarrojo monocromo. El modelo")
    print("  tiene que funcionar en dos dominios visuales muy distintos,")
    print("  y el aumento de datos por color (HSV) solo afecta a uno.")

# Lo que este análisis deja decidido

| hallazgo | consecuencia |
|---|---|
| las particiones son disjuntas y `trans_*` son cámaras nuevas | la partición **es** el experimento del entregable 3 |
| el desbalance entre especies es de cientos de veces | el mAP macro necesita acompañarse del ponderado por soporte |
| la mayoría de los animales ocupa <5% del cuadro | la resolución de entrada es una decisión real, no un default |
| la mezcla de especies **cambia** entre `cis` y `trans` | hay que controlarla antes de atribuir la brecha a la ubicación |
| casi la mitad de las imágenes es infrarrojo monocromo | el aumento por color actúa sobre la mitad del dataset |